In [24]:
import pandas as pd
import numpy as np
from datetime import date,datetime
from rapidfuzz import process, fuzz
from geopy.distance import geodesic
import warnings
import copy

warnings.filterwarnings('ignore')

In [2]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [33]:
# database File
df=pd.read_csv('elvisbuffalony2024obweekday_export_odbc.csv')
# KingElvis Dataframe
ke_df=pd.read_excel("Buffalo_NY_OB_KINGElvis.xlsx",sheet_name='Elvis_Review')
# Details File Stops Sheet
detail_df=pd.read_excel('details_buffalo_excel_template (1).xlsx',sheet_name='STOPS')

In [34]:
df.head(2)

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_DATE,ELVIS_USER_CHANGE_7_C_FIELD,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,5746,2024-09-17 06:50:07,75,en,2024-09-17 06:13:47,2024-09-17 06:50:07,172.59.176.185,https://buffalo-od.etc-research.com/,-25200,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,5747,2024-09-17 06:50:08,75,en,2024-09-17 06:13:49,2024-09-17 06:50:08,107.77.224.203,https://buffalo-od.etc-research.com/,-25200,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
df=df[df['INTERV_INIT']!='999']
df=df[df['HAVE_5_MIN_FOR_SURVECode']==1]
ke_df=ke_df[ke_df['INTERV_INIT']!='999']
ke_df=ke_df[ke_df['INTERV_INIT']!=999]
ke_df=ke_df[ke_df['HAVE_5_MIN_FOR_SURVECode']==1]

In [36]:
ke_df.head(2)

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1,RECORD_INFO
78,2024-09-25,5907,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,2024-09-25,5914,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
detail_df.head(2)

,agency,gtfs_ver,gtfs_date,route_short_name,route_long_name,direction_id,route_dir,seq_fixed,stop_id,stop_name,stop_lat,stop_lon,stop_lat6,stop_lon6,ETC_STOP_ID,ETC_STOP_NAME,notes,ETC_ROUTE_ID,ETC_ROUTE_NAME
0,NFT,1,8/26/2024,1,1,0,1_0,1.0,54240,Union Road Appletree Transit Center,42.888603,-78.751305,42.888603,-78.751305,NFT_1_1_00_54240,Union Road Appletree Transit Center,NaN,NFT_1_1_00,1 - WILLIAM (Inbound)
1,NFT,1,8/26/2024,1,1,0,1_0,2.0,54390,Union Road 2770 Union Road South,42.887570,-78.754747,42.887570,-78.754747,NFT_1_1_00_54390,Union Road 2770 Union Road South,NaN,NFT_1_1_00,1 - WILLIAM (Inbound)


In [38]:
stop_on_column_check=['stoponaddr']
stop_off_column_check=['stopoffaddr']
stop_on_id_column_check=['stoponclntid']
stop_off_id_column_check=['stopoffclntid']
stop_on_id_column=check_all_characters_present(df,stop_on_id_column_check)
stop_off_id_column=check_all_characters_present(df,stop_off_id_column_check)
stop_on_column=check_all_characters_present(df,stop_on_column_check)
stop_off_column=check_all_characters_present(df,stop_off_column_check)
stop_off_column,stop_on_column,stop_off_id_column,stop_on_id_column

(['STOP_OFF_ADDR'], ['STOP_ON_ADDR'], ['STOP_OFF_CLNTID'], ['STOP_ON_CLNTID'])

In [39]:
stop_on_lat_lon_columns_check=['stoponlat','stoponlong']
stop_off_lat_lon_columns_check=['stopofflat','stopofflong']
stop_on_lat_lon_columns=check_all_characters_present(df,stop_on_lat_lon_columns_check)
stop_off_lat_lon_columns=check_all_characters_present(df,stop_off_lat_lon_columns_check)
stop_off_lat_lon_columns.sort()
stop_on_lat_lon_columns.sort()
stop_off_lat_lon_columns,stop_on_lat_lon_columns

(['STOP_OFF_LAT', 'STOP_OFF_LONG'], ['STOP_ON_LAT', 'STOP_ON_LONG'])

In [40]:
columns_to_add = ['id', *stop_off_column, *stop_on_column, *stop_off_id_column, *stop_on_id_column,*stop_off_lat_lon_columns,*stop_on_lat_lon_columns]
columns_to_add

['id',
 'STOP_OFF_ADDR',
 'STOP_ON_ADDR',
 'STOP_OFF_CLNTID',
 'STOP_ON_CLNTID',
 'STOP_OFF_LAT',
 'STOP_OFF_LONG',
 'STOP_ON_LAT',
 'STOP_ON_LONG']

In [41]:
# Merge without prefixes or suffixes
ke_df = pd.merge(ke_df, df[columns_to_add], on='id', how='left')

In [42]:
ke_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,ADMIN_APPROVED.1,RECORD_INFO,STOP_OFF_ADDR,STOP_ON_ADDR,STOP_OFF_CLNTID,STOP_ON_CLNTID,STOP_OFF_LAT,STOP_OFF_LONG,STOP_ON_LAT,STOP_ON_LONG
0,2024-09-25,5907,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Kenmore Avenue Elmwood Avenue West,Main Street University Station Stat,NFT_1_5_00_26520,NFT_1_5_00_31360,42.958650,-78.878917,42.954526,-78.820463
1,2024-09-25,5914,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Niagara Street Albany Street South,Kenmore Avenue Eugene Avenue West,NFT_1_5_00_35530,NFT_1_5_00_26560,42.913200,-78.899739,42.958727,-78.873164
2,2024-09-25,5917,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Main Street Robie Street South,Main Street Winspear Avenue South,NFT_1_8_00_31240,NFT_1_8_00_31670,42.929717,-78.849574,42.951627,-78.825567
3,2024-09-25,5920,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Pearl Street Court Street South Far,Main Street Winspear Avenue South,NFT_1_8_00_42010,NFT_1_8_00_31670,42.885851,-78.875607,42.951627,-78.825567
4,2024-09-25,5922,HereAPI,Priyanka,Use,NaN,NaN,Elvis_Review,Elvis_Review,NaN,...,NaN,NaN,Niagara Street West Delavan Avenue,Ontario Street Chadduck Avenue Sout,NFT_1_5_00_35740,NFT_1_5_00_40040,42.922333,-78.898397,42.955496,-78.898445


In [43]:
route_surveyed_column_check=['routesurveyedcode']
route_surveyed_column=check_all_characters_present(ke_df,route_surveyed_column_check)
route_surveyed_column

['ROUTE_SURVEYEDCode']

In [44]:
ke_df[[route_surveyed_column[0]]].head(2)

,ROUTE_SURVEYEDCode
0,NFT_1_5_00
1,NFT_1_5_00


In [45]:
# required_ids=[6138,6288,6406,6464,6561,6576,6804,7018]
required_ids=[6119,6288]

In [46]:
# new_df=ke_df.iloc[:,:]
new_df=ke_df.query('id in @required_ids')

In [49]:
new_df['ROUTE_SURVEYEDCode_SPLITED']=new_df['ROUTE_SURVEYEDCode'].apply(lambda x : '_'.join(str(x).split('_')[0:-1]))
new_df[['ROUTE_SURVEYEDCode_SPLITED']]

,ROUTE_SURVEYEDCode_SPLITED
65,NFT_1_8
127,NFT_1_19


In [50]:
a='NFT_1_5_00'
'_'.join(a.split('_')[0:-1])

'NFT_1_5'

In [51]:
detail_df['ETC_ROUTE_ID_SPLITED']=detail_df['ETC_ROUTE_ID'].apply(lambda x : '_'.join(str(x).split('_')[0:-1]))
detail_df[['ETC_ROUTE_ID_SPLITED']].head(2)

,ETC_ROUTE_ID_SPLITED
0,NFT_1_1
1,NFT_1_1


In [52]:
detail_df[detail_df['ETC_ROUTE_ID_SPLITED']=='NFT_1_19']

,agency,gtfs_ver,gtfs_date,route_short_name,route_long_name,direction_id,route_dir,seq_fixed,stop_id,stop_name,stop_lat,stop_lon,stop_lat6,stop_lon6,ETC_STOP_ID,ETC_STOP_NAME,notes,ETC_ROUTE_ID,ETC_ROUTE_NAME,ETC_ROUTE_ID_SPLITED
1156,NFT,1,8/26/2024,19,19,0,19_0,1.0,31350,Main Street University Station Stat,42.955020,-78.820225,42.955020,-78.820225,NFT_1_19_00_31350,Main Street University Station Stat,NaN,NFT_1_19_00,19 - BAILEY (Inbound),NFT_1_19
1157,NFT,1,8/26/2024,19,19,0,19_0,2.0,29730,Main St Kenmore Ave East,42.957602,-78.818517,42.957602,-78.818517,NFT_1_19_00_29730,Main St Kenmore Ave East,NaN,NFT_1_19_00,19 - BAILEY (Inbound),NFT_1_19
1158,NFT,1,8/26/2024,19,19,0,19_0,3.0,30500,Main Street University Plaza East O,42.958092,-78.817232,42.958092,-78.817232,NFT_1_19_00_30500,Main Street University Plaza East O,NaN,NFT_1_19_00,19 - BAILEY (Inbound),NFT_1_19
1159,NFT,1,8/26/2024,19,19,0,19_0,4.0,31652,Goodyear Rd Bailey Ave East,42.958520,-78.814931,42.958520,-78.814931,NFT_1_19_00_31652,Goodyear Rd Bailey Ave East,NaN,NFT_1_19_00,19 - BAILEY (Inbound),NFT_1_19
1160,NFT,1,8/26/2024,19,19,0,19_0,5.0,2360,Bailey Avenue 3675 Bailey Avenue So,42.956654,-78.813956,42.956654,-78.813956,NFT_1_19_00_2360,Bailey Avenue 3675 Bailey Avenue So,NaN,NFT_1_19_00,19 - BAILEY (Inbound),NFT_1_19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1283,NFT,1,8/26/2024,19,19,1,19_1,63.0,3070,Bailey Avenue Main Street North,42.957917,-78.813754,42.957917,-78.813754,NFT_1_19_01_3070,Bailey Avenue Main Street North,NaN,NFT_1_19_01,19 - BAILEY (Outbound),NFT_1_19
1284,NFT,1,8/26/2024,19,19,1,19_1,64.0,29740,Main Street Callodine Avenue South,42.959163,-78.814817,42.959163,-78.814817,NFT_1_19_01_29740,Main Street Callodine Avenue South,NaN,NFT_1_19_01,19 - BAILEY (Outbound),NFT_1_19
1285,NFT,1,8/26/2024,19,19,1,19_1,65.0,30510,Main Street University Plaza West N,42.958484,-78.816490,42.958484,-78.816490,NFT_1_19_01_30510,Main Street University Plaza West N,NaN,NFT_1_19_01,19 - BAILEY (Outbound),NFT_1_19
1286,NFT,1,8/26/2024,19,19,1,19_1,66.0,31640,Main Street Kenmore Ave West,42.957350,-78.819062,42.957350,-78.819062,NFT_1_19_01_31640,Main Street Kenmore Ave West,NaN,NFT_1_19_01,19 - BAILEY (Outbound),NFT_1_19


In [53]:
def get_distance_between_coordinates(lat1, lon1, lat2, lon2):
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
        
        coords_1 = (lat1, lon1)
        coords_2 = (lat2, lon2)
        
        distance = geodesic(coords_1, coords_2).miles
        return distance
    except (ValueError, TypeError) as e:
        # Handle the exception here
        print(f"Error calculating distance: {e}")  # Change to the desired distance unit

In [55]:
stop_on_lat_lon_columns

['STOP_ON_LAT', 'STOP_ON_LONG']

In [56]:
# for _, row in new_df.iterrows():
#     nearest_stop_seq = []
    
#     stop_on_lat = row[stop_on_lat_lon_columns[0]]
#     stop_on_long = row[stop_on_lat_lon_columns[1]]
    
#     # Filtered data
#     filtered_df = detail_df[detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED']][
#         ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID']
#     ]
    
#     # List to store distances
#     distances = []
    
#     # Calculate distances for all rows in filtered_df
#     for _, detail_row in filtered_df.iterrows():
#         stop_lat6 = detail_row['stop_lat6']
#         stop_lon6 = detail_row['stop_lon6']
        
#         # Compute distance
#         distance = get_distance_between_coordinates(stop_on_lat, stop_on_long, stop_lat6, stop_lon6)
#         distances.append((distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID']))
    
#     # Find the nearest stop (minimum distance)
#     if distances:
#         nearest_stop = min(distances, key=lambda x: x[0])  # x[0] is the distance
#         nearest_stop_seq.append(nearest_stop)
    
#     # Process nearest_stop_seq as needed
#     print(f"Nearest stop details for row: {nearest_stop_seq}")


Nearest stop details for row: [(0.0, 23.0, 'NFT_1_8_01_31553')]
Nearest stop details for row: [(0.0, 65.0, 'NFT_1_19_00_2530')]


# get the value of seq_fixed ETC_STOP_ID ETC_STOP_NAME using NEAREST stop

In [58]:
# Iterate through new_df rows
for _, row in new_df.iterrows():
    nearest_stop_seq = []
    
    stop_on_id=row[stop_on_id_column[0]]    
    
    stop_on_lat = row[stop_on_lat_lon_columns[0]]
    stop_on_long = row[stop_on_lat_lon_columns[1]]
    
    # Filtered data
    filtered_df = detail_df[detail_df['ETC_ROUTE_ID_SPLITED'] == row['ROUTE_SURVEYEDCode_SPLITED']][
        ['stop_lat6', 'stop_lon6', 'seq_fixed', 'ETC_STOP_ID','ETC_STOP_NAME']
    ]
    
    # List to store distances
    distances = []
    
    # Calculate distances for all rows in filtered_df
    for _, detail_row in filtered_df.iterrows():
        stop_lat6 = detail_row['stop_lat6']
        stop_lon6 = detail_row['stop_lon6']
        
        # Compute distance
        distance = get_distance_between_coordinates(stop_on_lat, stop_on_long, stop_lat6, stop_lon6)
        
        # Skip distance if it is 0
        if distance == 0:
            continue
        
        distances.append((distance, detail_row['seq_fixed'], detail_row['ETC_STOP_ID'],detail_row['ETC_STOP_NAME']))
    
    # Find the nearest stop (minimum distance)
    if distances:
        nearest_stop = min(distances, key=lambda x: x[0])  # x[0] is the distance
        nearest_stop_seq.append(nearest_stop)
    
    # Process nearest_stop_seq as needed
    print(f"Nearest stop details for row: {nearest_stop_seq}")

Nearest stop details for row: [(0.012625893194940766, 32.0, 'NFT_1_8_00_31550', 'Main Street West Utica Street South')]
Nearest stop details for row: [(0.16111940448485781, 2.0, 'NFT_1_19_01_2450', 'Bailey Avenue 97 Bailey Avenue Nort')]


In [28]:
def get_nearest_distance_stop_sequence(row):
    pass

In [16]:
# for _,row in new_df.iterrows():
#     stop_on_id=row[stop_on_id_column[0]]    
#     stop_off_id=row[stop_off_id_column[0]]
#     stop_on_addr=row[stop_on_column[0]]    
#     stop_off_addr=row[stop_off_column[0]]
#     route_surveyed=row[route_surveyed_column[0]]
#     print(stop_on_addr,stop_off_addr,route_surveyed)
# #     alight_seq_df = detail_df[(detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
# #                        (detail_df['ETC_STOP_ID'] == stop_off_id)][['seq_fixed']]

# # updated code using ETC_STOP_NAME to get seq_fixed value
#     alight_seq_df = detail_df[(detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
#                    (detail_df['ETC_STOP_ID'] == stop_off_addr)][['seq_fixed']]
#     if not alight_seq_df.empty:
#         alight_seq = alight_seq_df.iloc[0, 0]
#     else:
#         alight_seq = 0
        
# #     board_seq_df = detail_df[(detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
# #                           (detail_df['ETC_STOP_ID'] == stop_on_id)][['seq_fixed']]
# # updated code using ETC_STOP_NAME to get seq_fixed value
#     board_seq_df = detail_df[(detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
#                       (detail_df['ETC_STOP_NAME'] == stop_on_addr)][['seq_fixed']]
#     if not board_seq_df.empty:
#         board_seq = board_seq_df.iloc[0, 0]
#     else:
#         board_seq = 0
        
#     new_df.loc[row.name, 'Alight_Seq'] = alight_seq
#     new_df.loc[row.name, 'Board_Seq'] = board_seq

Eggert Road Janet Street South nan NFT_1_12_00
Bailey Avenue Abbott Road Static Bailey Avenue Gerald Place North Kenfield NFT_1_19_01
Bailey Avenue Winspear Avenue South nan NFT_1_19_00
Main Street University Station Stat nan NFT_1_12_01
North Division Ellicott Street West nan NFT_1_5_01
Pearl Street Seneca Street South 400 Exchange St Suite #3 NFT_1_8_00
Ferry Street Richmond Avenue East nan NFT_1_12_01
Genesee Street Oak Street East Genesee Street Latour Street East NFT_1_24_01


In [ ]:
# Fuzz Search Implemented in below cell

In [35]:
for _, row in new_df.iterrows():
    stop_on_id = row[stop_on_id_column[0]]    
    stop_off_id = row[stop_off_id_column[0]]
    stop_on_addr = row[stop_on_column[0]]    
    stop_off_addr = row[stop_off_column[0]]
    route_surveyed = row[route_surveyed_column[0]]
    
    print(stop_on_addr, stop_off_addr, route_surveyed)
    
    # Using ETC_STOP_NAME to get seq_fixed value for alight_seq_df
    alight_seq_df = detail_df[
        (detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
        (detail_df['ETC_STOP_NAME'].str.lower() == str(stop_off_addr).lower())
    ][['seq_fixed']]
    
    # Apply fuzzy matching for alight_seq_df if empty
    if alight_seq_df.empty:
        best_match, score, _ = process.extractOne(
            str(stop_off_addr).lower(),
            detail_df[detail_df['ETC_ROUTE_ID'] == route_surveyed]['ETC_STOP_NAME'].str.lower(),
            scorer=fuzz.ratio
        )
        if score >= 40:  # Threshold for fuzzy matching
            alight_seq_df = detail_df[
                (detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
                (detail_df['ETC_STOP_NAME'].str.lower() == best_match)
            ][['seq_fixed']]
    
    alight_seq = alight_seq_df.iloc[0, 0] if not alight_seq_df.empty else 0
    
    # Using ETC_STOP_NAME to get seq_fixed value for board_seq_df
    board_seq_df = detail_df[
        (detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
        (detail_df['ETC_STOP_NAME'].str.lower() == str(stop_on_addr).lower())
    ][['seq_fixed']]
    
    # Apply fuzzy matching for board_seq_df if empty
    if board_seq_df.empty:
        best_match, score, _ = process.extractOne(
            str(stop_on_addr).lower(),
            detail_df[detail_df['ETC_ROUTE_ID'] == route_surveyed]['ETC_STOP_NAME'].str.lower(),
            scorer=fuzz.ratio
        )
        if score >= 40:  # Threshold for fuzzy matching
            board_seq_df = detail_df[
                (detail_df['ETC_ROUTE_ID'] == route_surveyed) & 
                (detail_df['ETC_STOP_NAME'].str.lower() == best_match)
            ][['seq_fixed']]
    
    board_seq = board_seq_df.iloc[0, 0] if not board_seq_df.empty else 0
    
    # Update new_df with the results
    new_df.loc[row.name, 'Alight_Seq'] = alight_seq
    new_df.loc[row.name, 'Board_Seq'] = board_seq


Eggert Road Janet Street South nan NFT_1_12_00
Bailey Avenue Abbott Road Static Bailey Avenue Gerald Place North Kenfield NFT_1_19_01
Bailey Avenue Winspear Avenue South nan NFT_1_19_00
Main Street University Station Stat nan NFT_1_12_01
North Division Ellicott Street West nan NFT_1_5_01
Pearl Street Seneca Street South 400 Exchange St Suite #3 NFT_1_8_00
Ferry Street Richmond Avenue East nan NFT_1_12_01
Genesee Street Oak Street East Genesee Street Latour Street East NFT_1_24_01


In [36]:
new_df[['Alight_Seq','Board_Seq']]

,Alight_Seq,Board_Seq
79,0.0,19.0
127,45.0,1.0
163,0.0,9.0
187,0.0,74.0
235,0.0,2.0
244,39.0,49.0
345,0.0,10.0
434,29.0,12.0


In [18]:
new_df['SEQ_CHECK'] = new_df['Alight_Seq'] - new_df['Board_Seq']
new_df[['SEQ_CHECK']]

,SEQ_CHECK
79,-19.0
127,-1.0
163,-9.0
187,-74.0
235,-2.0
244,-49.0
345,-10.0
434,-12.0


In [19]:
new_df=new_df[new_df['SEQ_CHECK']<0]

In [20]:
# ids_list = []

# for _, row in new_df.iterrows():
#     if row['SEQ_CHECK'] < 0:
#         # Extract route code and modify it
#         ids_list.append(row['id'])
#         stop_on_id=row[stop_on_id_column[0]]    
#         stop_off_id=row[stop_off_id_column[0]]
#         route_code = row[route_surveyed_column[0]]
#         new_route_code = f"{'_'.join(route_code.split('_')[:-1])}_01" if route_code.split('_')[-1] == '00' else f"{'_'.join(route_code.split('_')[:-1])}_00"
        
#         # Get new route name
#         new_route_name_row = detail_df[detail_df['ETC_ROUTE_ID'] == new_route_code]
#         if not new_route_name_row.empty:
#             new_route_name = new_route_name_row['ETC_ROUTE_NAME'].iloc[0]
#         else:
#             # Handle the case when the new route name is not found
#             continue  # Skip to the next iteration if new route code is not valid

#         # Board stop details
#         board_stop_row = detail_df[(detail_df['ETC_ROUTE_ID'] == new_route_code) & (detail_df['ETC_STOP_ID'] == stop_on_id)]
#         if not board_stop_row.empty:
#             board_stop_code = board_stop_row['ETC_STOP_ID'].iloc[0]
#             board_stop_lat_lon = board_stop_row[['seq_fixed', 'stop_lat6', 'stop_lon6']].iloc[0]
#         else:
#             # Skip if no board stop information is found
#             continue

#         # Alight stop details
#         alight_stop_row = detail_df[(detail_df['ETC_ROUTE_ID'] == new_route_code) & (detail_df['ETC_STOP_NAME'] == stop_off_id)]
#         if not alight_stop_row.empty:
#             alight_stop_code = alight_stop_row['ETC_STOP_ID'].iloc[0]
#             alight_stop_lat_lon = alight_stop_row[['seq_fixed', 'stop_lat6', 'stop_lon6']].iloc[0]
#         else:
#             # Skip if no alight stop information is found
#             continue

#         # Update new_df with the retrieved values
#         new_df.loc[row.name, 'ROUTE_DESCRIPTION[CODE]'] = new_route_code
#         new_df.loc[row.name, 'ETC_ROUTE_NAME'] = new_route_name

#         new_df.loc[row.name, 'Board_Seq'] = board_stop_lat_lon['seq_fixed']
#         new_df.loc[row.name, 'BOARD_STOP[CODE]'] = board_stop_code

#         new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = board_stop_lat_lon['stop_lat6']
#         new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = board_stop_lat_lon['stop_lon6']

#         new_df.loc[row.name, 'Alight_Seq'] = alight_stop_lat_lon['seq_fixed']
#         new_df.loc[row.name, 'ALIGHT_STOP[CODE]'] = alight_stop_code

#         new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = alight_stop_lat_lon['stop_lat6']
#         new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = alight_stop_lat_lon['stop_lon6']


In [21]:
# ids_list=[]
# for idx in new_df.index:
#     if new_df.loc[idx, 'SEQ_CHECK'] < 0:
#         # Extract route code and modify it
#         ids_list.append(new_df.loc[idx, 'id'])
        
#         stop_on_id = new_df.loc[idx, stop_on_id_column[0]]  # Boarding
#         stop_off_id = new_df.loc[idx, stop_off_id_column[0]]  # Alighting
#         route_code = new_df.loc[idx, route_surveyed_column[0]]  # ROUTE_SURVEYEDCode

#         # Modify the route code
#         new_route_code = (
#             f"{'_'.join(route_code.split('_')[:-1])}_01" 
#             if route_code.split('_')[-1] == '00' 
#             else f"{'_'.join(route_code.split('_')[:-1])}_00"
#         )

#         # Get new route name
#         new_route_name_row = detail_df[detail_df['ETC_ROUTE_ID'] == new_route_code]
#         if not new_route_name_row.empty:
#             new_route_name = new_route_name_row['ETC_ROUTE_NAME'].iloc[0]
#         else:
#             print(f"New route code {new_route_code} not found in detail_df")
#             continue  # Skip this row if no new route name is found

#         # Update the DataFrame with the new values
# #         new_df.loc[idx, route_surveyed_column[0]] = new_route_code
# #         new_df.loc[idx, 'ROUTE_SURVEYED'] = new_route_name
#         new_df.loc[idx, 'ROUTE_SURVEYEDCode_New'] = new_route_code
#         new_df.loc[idx, 'ROUTE_SURVEYED_NEW'] = new_route_name


In [22]:
# len(ids_list)

In [23]:
# new_df.head()

In [24]:
# new_df[['id',route_surveyed_column[0],'ROUTE_SURVEYED']].to_csv('Reverse_directions.csv',index=False)

In [25]:
# new_df.to_csv('Reverse_directions_01.csv',index=False)

In [26]:
# new_df[[route_surveyed_column[0],'ROUTE_SURVEYEDCode_New','ROUTE_SURVEYED','ROUTE_SURVEYED_NEW']]

In [27]:
ids_list = []
for idx in new_df.index:
    if new_df.loc[idx, 'SEQ_CHECK'] < 0:
        # Extract route code and modify it
        ids_list.append(new_df.loc[idx, 'id'])
        
        stop_on_id = new_df.loc[idx, stop_on_id_column[0]]  # Boarding
        stop_off_id = new_df.loc[idx, stop_off_id_column[0]]  # Alighting
        route_code = new_df.loc[idx, route_surveyed_column[0]]  # ROUTE_SURVEYEDCode

        # Modify the route code
        new_route_code = (
            f"{'_'.join(route_code.split('_')[:-1])}_01" 
            if route_code.split('_')[-1] == '00' 
            else f"{'_'.join(route_code.split('_')[:-1])}_00"
        )

        # Set default values to retain previous values
        new_df.loc[idx, 'ROUTE_SURVEYEDCode_New'] = route_code
        new_df.loc[idx, 'ROUTE_SURVEYED_NEW'] = new_df.loc[idx, 'ROUTE_SURVEYED']

        # Get new route name
        new_route_name_row = detail_df[detail_df['ETC_ROUTE_ID'] == new_route_code]
        if not new_route_name_row.empty:
            new_route_name = new_route_name_row['ETC_ROUTE_NAME'].iloc[0]
            
            # Update the DataFrame with the new values
            new_df.loc[idx, 'ROUTE_SURVEYEDCode_New'] = new_route_code
            new_df.loc[idx, 'ROUTE_SURVEYED_NEW'] = new_route_name
        else:
            # Log and keep previous values
            print(f"New route code {new_route_code} not found in detail_df. Keeping previous values.")


In [31]:
new_df[['id',route_surveyed_column[0],'ROUTE_SURVEYEDCode_New','ROUTE_SURVEYED','ROUTE_SURVEYED_NEW']]

,id,ROUTE_SURVEYEDCode,ROUTE_SURVEYEDCode_New,ROUTE_SURVEYED,ROUTE_SURVEYED_NEW
79,6138,NFT_1_12_00,NFT_1_12_01,12 - UTICA (Inbound),12 - UTICA (Outbound)
127,6288,NFT_1_19_01,NFT_1_19_00,19 - BAILEY (Outbound),19 - BAILEY (Inbound)
163,6406,NFT_1_19_00,NFT_1_19_01,19 - BAILEY (Inbound),19 - BAILEY (Outbound)
187,6464,NFT_1_12_01,NFT_1_12_00,12 - UTICA (Outbound),12 - UTICA (Inbound)
235,6561,NFT_1_5_01,NFT_1_5_00,5 - NIAGARA-KENMORE (Outbound),5 - NIAGARA-KENMORE (Inbound)
244,6576,NFT_1_8_00,NFT_1_8_01,8 - MAIN (Inbound),8 - MAIN (Outbound)
345,6804,NFT_1_12_01,NFT_1_12_00,12 - UTICA (Outbound),12 - UTICA (Inbound)
434,7018,NFT_1_24_01,NFT_1_24_00,24 - GENESEE (Outbound),24 - GENESEE (Inbound)


In [29]:
# new_df.to_csv('Reverse_directions_02.csv',index=False)

In [30]:
trtetet

NameError: name 'trtetet' is not defined

In [ ]:
required_ids=[6138,6288,6406,6464,6561,6576,6804,7018]

In [ ]:
test_df=df.query('id in @required_ids')

In [ ]:
boarding_columns_checks=['prevtran1onbuslat', 'prevtran1onbuslong',
                         'prevtran2onbuslat', 'prevtran2onbuslong',
                         'prevtran3onbuslat', 'prevtran3onbuslong', 
                         'prevtran4onbuslat', 
                         'prevtran4onbuslong','stoponlat', 'stoponlong',
                         'stopofflat', 'stopofflong',
                          'nexttran1offbuslat','nexttran1offbuslong',  
                         'nexttran2offbuslat', 'nexttran2offbuslong', 
                          'nexttran3offbuslat', 'nexttran3offbuslong', 
                          'nexttran4offbuslat', 'nexttran4offbuslong',]
boarding_columns=check_all_characters_present(df,boarding_columns_checks)
boarding_columns.sort()
boarding_columns

In [ ]:
boarding_alighting_addr_columns_check=['stoponadd','stopoffadd','']

In [ ]:
origin_destin_columns_checks=['originaddresslat','originaddresslong', 'destinaddresslat', 'destinaddresslong']
origin_destin_columns=check_all_characters_present(df,origin_destin_columns_checks)
origin_destin_columns.sort()
origin_destin_columns

In [ ]:
test_df[['id',*boarding_columns]].head()

In [ ]:
for index, row in df.iterrows():

    if not pd.isna(row[boarding_columns[18]]) and not pd.isna(row[boarding_columns[19]]):
        #  'STOP_ON_LAT',
        # 'STOP_ON_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[18]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[19]]        
    elif not pd.isna(row[boarding_columns[8]]) and not pd.isna(row[boarding_columns[9]]):
        #'PREV_TRAN_1_ON_BUS_LAT',
        #'PREV_TRAN_1_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[8]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[9]]       
    elif not pd.isna(row[boarding_columns[10]]) and not pd.isna(row[boarding_columns[11]]):
        #  'PREV_TRAN_2_ON_BUS_LAT',
        # 'PREV_TRAN_2_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[10]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[11]]
    elif not pd.isna(row[boarding_columns[12]]) and not pd.isna(row[boarding_columns[13]]):
        #  'PREV_TRAN_3_ON_BUS_LAT',
        # 'PREV_TRAN_3_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[12]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[13]]
    elif not pd.isna(row[boarding_columns[14]]) and not pd.isna(row[boarding_columns[15]]):
        #  'PREV_TRAN_4_ON_BUS_LAT',
        # 'PREV_TRAN_4_ON_BUS_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[14]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[15]]
    elif not pd.isna(row[boarding_columns[18]]) and not pd.isna(row[boarding_columns[19]]):
        #  'STOP_ON_LAT',
        # 'STOP_ON_LONG'
        df.loc[index, 'FIRST_BOARDING_LAT'] = row[boarding_columns[18]]
        df.loc[index, 'FIRST_BOARDING_LONG'] = row[boarding_columns[19]]
    else:
        df.loc[index, 'FIRST_BOARDING_LAT'] = np.nan
        df.loc[index, 'FIRST_BOARDING_LONG'] = np.nan
    #      
    if not pd.isna(row[boarding_columns[6]]) and not pd.isna(row[boarding_columns[7]]):
        #  'NEXT_TRAN_4_OFF_BUS_LAT',
        # 'NEXT_TRAN_4_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[6]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[7]]
    elif not pd.isna(row[boarding_columns[4]]) and not pd.isna(row[boarding_columns[5]]):
        #  'NEXT_TRAN_3_OFF_BUS_LAT',
        # 'NEXT_TRAN_3_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[4]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[5]]
    elif not pd.isna(row[boarding_columns[2]]) and not pd.isna(row[boarding_columns[3]]):
        #  'NEXT_TRAN_2_OFF_BUS_LAT',
        # 'NEXT_TRAN_2_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[2]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[3]]
    elif not pd.isna(row[boarding_columns[0]]) and not pd.isna(row[boarding_columns[1]]):
        #  'NEXT_TRAN_1_OFF_BUS_LAT',
        # 'NEXT_TRAN_1_OFF_BUS_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[0]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[1]]
    elif not pd.isna(row[boarding_columns[16]]) and not pd.isna(row[boarding_columns[17]]):
        #  'STOP_OFF_LAT',
        # 'STOP_OFF_LONG'
        df.loc[index, 'LAST_ALIGHTING_LAT'] = row[boarding_columns[16]]
        df.loc[index, 'LAST_ALIGHTING_LONG'] = row[boarding_columns[17]]
    else:
        df.loc[index, 'LAST_ALIGHTING_LAT'] = np.nan
        df.loc[index, 'LAST_ALIGHTING_LONG'] = np.nan